In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage, BaseMessage
from typing import TypedDict,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
import operator

In [2]:
load_dotenv()

llm = ChatOpenAI()

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = MemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Why did the pizza go to the therapist?\nBecause it had too many toppings and couldn't hold it all together!",
 'explanation': 'This joke plays on the idea of a pizza having too many toppings and not being able to hold them all together, both physically and mentally. By going to a therapist, the pizza is seeking help in managing the overwhelming situation it is facing. It combines the concept of a therapy session with a humorous twist on the struggles of a pizza with too many toppings.'}

In [9]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the spaghetti break up with the lasagna? \nBecause it couldn't handle the cheesy commitment!",
 'explanation': 'This joke is a play on words, using the idea of food items breaking up in a humorous way. In this case, the spaghetti broke up with the lasagna because it "couldn\'t handle the cheesy commitment." This is a pun on the word "cheesy," which can mean something that is overly romantic or sentimental, as well as referring to the cheese commonly used in lasagna. The joke implies that the lasagna was too emotionally intense or committed for the spaghetti to handle, leading to their breakup.'}